# 01/07 贝叶斯定理与统计思维 — 学习 Notebook

**阅读全译本**：[../docs/zh.md](../docs/zh.md)　 **英文原文**：[../docs/en.md](../docs/en.md)

**学习状态：待开始。** 本 Notebook 用于保留你后续独立提交的代码、数学证明、实际测试与导师讲评。正式测验尚未进行，不能提前标记课程完成。

课程主线：**问题 → 概念 → Build It → Use It → Ship It → Exercises → Post Quiz**。官方答案文件 `../quiz.json` 含正确答案，建议正式后测前暂不查看。

参考的可运行官方程序：[bayes.py](bayes.py)。导师不预先填写需要你独立完成的答案。

## 热身：衔接上一课 01/06

上一课你独立推导了条件概率公式，掌握了“两个条件概率方向不能随意互换”。

**思考（先不查答案）**：已知 $P(A\mid B)$ 与 $P(B\mid A)$，什么时候二者会相等？只知道其中一个通常能算出另一个吗？

第一段教学将从 **$P(A\cap B)$ 的两种展开方式**开始，推导贝叶斯定理，而不是直接记忆公式。

## 概念工作区：先验、似然、证据和后验

请在后续课堂中写出两种联合概率展开、证明贝叶斯定理，并用“疾病罕见、99% 灵敏度、1% 假阳性率”的例子说明为什么 $P(\text{sick}\mid\text{positive})$ 并不是 99%。

本节预留作为你的原始推导、疑问和纠错记录。

In [ ]:
# 数值实验工作区：先由你独立实现 bayes(...)，再和官方代码进行对照。
# 不在此处预填练习答案。
import math


## Build It：朴素贝叶斯垃圾邮件分类器

- 在给定类别时，为什么能够把特征的条件似然相乘？
- 未见词的概率为什么可能为零？拉普拉斯平滑怎样处理？
- 为什么要使用 log 概率而不是很多概率直接相乘？
- 实现后再与官方 [bayes.py](bayes.py) 以及 scikit-learn 的 `MultinomialNB` 对照。

**你的独立代码和测试将追加在这里。**

## MLE / MAP、Beta–Binomial 与 A/B 测试

等待依次学习：MLE 与 MAP 的不同、Gaussian 先验与 L2 正则化、Beta 共轭先验、顺序与批次更新的等价关系、A/B 测试中的后验比较。

**官方四道练习**：两次阳性检测、平滑系数变化、加入邮件长度特征、Beta(2,2) 先验下的 MAP 估计。

学生提交后再添加可复现实验和讲评。

## 官方后测与课程总结（待完成）

原版 [quiz.json](../quiz.json) 的后测共有 3 题；必须按顺序逐题回答并给出评分，最后才更新仓库的 `LEARNING.md`。

下一步：读 [../docs/zh.md](../docs/zh.md) 的“问题”“概念”部分，开始 01/07 的教学。

## 第 1 讲：从条件概率到贝叶斯推理——问题与上一课热身

**课程状态：已开始教学，等待学生回答热身，不计分。**

官方 01/07 的核心问题：某疾病患病率为每一万人 1 人；检测的灵敏度为 99%，假阳性率为 1%。某人检测为阳性，实际患病的后验概率是多少？

**两个容易混淆的条件方向：**
- 问题 A：**已经患病**的前提下，得到**阳性结果**的概率；
- 问题 B：**已经得到阳性结果**的前提下，**实际患病**的概率。

直觉上，它们不是同一个问题。一个阳性结果既可能来自真正患病者的真阳性，也可能来自庞大健康人群中少量的假阳性。必须考虑疾病的先验基准率以及两类人的比例，而不能见到“灵敏度 99%”就直接认定“阳性后患病概率 99%”。

**本轮暂停在“问题 → 概念”的直觉阶段**，后续导师将从联合概率的两种分解推导贝叶斯公式，使用 100 万人的自然频数表验证，并逐步完成官方 Build It、Use It、Ship It、四道 Exercises 和 3 道 post quiz。

### 热身问题（继承 01/06，等待学生作答，不计分）

1. 单标签交叉熵 $L=-\ln p_y$：对同一真实类别预测概率为 $0.9$ 与 $0.01$ 时，哪种预测损失更大？为什么？
2. 语言模型的 token 概率连乘很小：为什么改在对数域求和？得到的总 log probability 在数值上有限，是否意味着用 $\exp$ 恢复原始概率永远不会下溢？

**指导原则：** 在学生回答前不提前给出答案，不替学生填写手写代码。收到回答后保存学生原话、数学讲评、可运行例子和下一道题。

## 第 1 讲续：热身讲评（2/2，非正式测验）

学生原话：“问题1 0.01的损失更大,因为负对数似然是单调递减的,因为log probalilities可以把连乘转换成加法2,不能保证,”

### 第一题：One-hot 分类交叉熵

**正确。** 令 $L(p_y)=-\ln p_y$，有

$$\frac{dL}{dp_y}=-\frac{1}{p_y}<0\quad(0<p_y\le1).$$

因此 $p_y=0.01$ 时损失 $4.605170\ldots$，高于 $p_y=0.9$ 时的 $0.105360\ldots$。

术语澄清：**$\ln p$ 严格递增，$-\ln p$ 严格递减**，学生指的是负对数损失，判断无误。

### 第二题：为什么对数域计算不代表恢复原始概率永远不会下溢？

**正确。** $\ln\prod_i p_i=\sum_i\ln p_i$，在 log 域求和可以避免很多极小概率在相乘阶段下溢。但在总 log probability 很小时，最终调用 $\exp(\sum_i\ln p_i)$，仍然可能发生浮点下溢为 $0.0$。数学上的正概率与浮点零必须区分。

这两题是不计分的上一课知识复习，不影响 01/07 未来官方 post quiz 的评分。

## 第 2 讲：从联合概率严格推导 Bayes' Theorem

### 先区分“条件方向”

假设 $D$ 表示**患病**，$+$ 表示**检测阳性**。

- $P(+\mid D)$：在患病者中，检测出现阳性的概率，即**灵敏度**。
- $P(D\mid +)$：在阳性者中，实际上患病的概率，即**阳性预测值/后验概率**。

两者不可以直接交换，因为条件概率的分母不同：

$$P(D\mid +)=\frac{P(D\cap +)}{P(+)},\qquad
P(+\mid D)=\frac{P(D\cap +)}{P(D)}.$$

两个式子都包含同一个联合概率 $P(D\cap +)$，因此

$$P(D\mid +)P(+)=P(+\mid D)P(D).$$

整理：

$$\boxed{P(D\mid +)=\frac{P(+\mid D)P(D)}{P(+)}}.$$

对于“检测阳性”，考虑全部可能来源——**患者中的真阳性**与**健康者中的假阳性**。全概率公式给出

$$P(+)=P(+\mid D)P(D)+P(+\mid\neg D)P(\neg D).$$

于是二元假设下的贝叶斯公式：

$$\boxed{P(D\mid +)=
\frac{P(+\mid D)P(D)}
{P(+\mid D)P(D)+P(+\mid\neg D)(1-P(D))}}.$$

- $P(D)$：先验 Prior，即患病率；
- $P(+\mid D)$：似然 Likelihood，即灵敏度；
- $P(+)$：证据 Evidence，即总体阳性概率；作为归一化分母；
- $P(D\mid +)$：后验 Posterior，观察阳性后的患病概率。

### 100 万人的自然频数实验（不使用抽样误差）

课文条件：$P(D)=0.0001$，$P(+\mid D)=0.99$，$P(+\mid\neg D)=0.01$。

想象恰好统计 **1,000,000 人**，并用上述比例计算期望人数：

| 人群 | 全部人数 | 检测阳性 | 检测阴性 |
|---|---:|---:|---:|
| 患病 | 100 | 99（真阳性） | 1 |
| 健康 | 999,900 | 9,999（假阳性） | 989,901 |
| 合计 | 1,000,000 | **10,098** | 989,902 |

在所有检测阳性的 $10,098$ 人中，只有 $99$ 人真正患病，因此

$$P(D\mid +)=\frac{99}{10098}=0.0098039215686\ldots
\approx \boxed{0.9804\%}.$$

用概率公式计算完全一致：

$$P(+)=0.99\cdot0.0001+0.01\cdot0.9999=0.010098,$$

$$P(D\mid +)=\frac{0.99\cdot0.0001}{0.010098}
\approx0.0098039215686.$$

**重要假设**：“检测 99% 准确”并不单独决定后验；上面的算例明确假定患者灵敏度为 99%，健康者假阳性率为 1%。用于临床推断还需真实患病率与实际检测性能等信息。

### 导师独立运行的数字验证（非学生手写 Bayes 函数）

本单元检验概率法与自然频数法数值一致。实际由导师 Python 运行得到：`evidence=0.010098`、`posterior=0.009803921569`，即 `0.980392%`。

In [ ]:
import math

prior = 0.0001
sensitivity = 0.99
false_positive_rate = 0.01

evidence = sensitivity * prior + false_positive_rate * (1 - prior)
posterior = sensitivity * prior / evidence

population = 1_000_000
sick = 100
healthy = 999_900
true_positives = 99
false_positives = 9_999

assert math.isclose(evidence,0.010098,rel_tol=1e-12)
assert math.isclose(posterior,99/10098,rel_tol=1e-12)
assert true_positives + false_positives == 10_098
assert sick + healthy == population

print(f"Evidence P(positive): {evidence:.10f}")
print(f"Posterior P(sick|positive): {posterior:.12f}")
print(f"Positive predictive value: {100*posterior:.6f}%")

## 第 2 讲理解检查（等待学生回答）

当患病率保持 $P(D)=0.0001$、灵敏度保持 $P(+\mid D)=0.99$ 不变，仅把健康者的**假阳性率从 1% 降至 0.1%**（$0.01\rightarrow0.001$）时：

1. 得到阳性结果后，真正患病的概率会**上升还是下降**？请用贝叶斯公式解释分母会怎样变化。
2. 尝试算出新的 $P(D\mid +)$，可以保留到百分比小数点后两位。

学生回答后，导师再验收并进入官方 Build It 第 1 步：由学生独立实现 `bayes(prior, likelihood, false_positive_rate)`。这里暂时**不提前填写完整函数答案**。

## 第 2 讲理解检查：学生解答与讲评（2/2）

**学生原始思路**：“问题1 检测到阳性,真正患病 是p(D|+) = [贝叶斯二元公式] 分子不变, 分母减少,整体增加,患病概率会升高, 问题2 是0.09”。

**判定：两题正确。** 第一问的推理严谨，明确看出了真阳性概率（分子）不变，健康人假阳性概率降低使整体阳性概率（分母）下降，所以检测阳性后的实际患病后验上升。第二问写为 `0.09`，应理解为**小数概率 0.09 = 9%**，与正确答案相符的近似值；若误写成 `0.09%` 则有 100 倍单位误差。

设患病率 $p=0.0001$、灵敏度 $s=0.99$、假阳性率 $f=0.001$，则

$$P(D\mid+)=\frac{sp}{sp+f(1-p)}
=\frac{0.000099}{0.000099+0.0009999}
=\frac{0.000099}{0.0010989}
\approx 0.09009009009.$$ 

以百分比表示为：

$$\boxed{P(D\mid+)\approx 9.01\%}.$$

同样的先验与灵敏度，若 $f=0.01$，原后验为 $0.00980392157\approx0.98\%$。由此可见降低假阳性率十倍，会让阳性预测值显著升高，但**并不是严格增加十倍**：因为分母中的真阳性项 $sp$ 是固定的。

### 用自然频数验证（1000 万人）

- 患病约 1000 人，其中检测阳性 $990$ 人；
- 健康约 $9,999,000$ 人，其中假阳性 $9,999$ 人；
- 所有阳性共 $990+9999=10,989$ 人，真正患病者占 $990/10,989\approx0.09009009$。

这里的数字是对一个理想化 1000 万人总体按理论比例计算的期望人数，真实抽样有波动。

**必须区分**：`0.09` 表示约 $9\%$，而 `0.09%` 表示小数概率 $0.0009$。在机器学习中混写小数概率与百分比是一种常见且影响严重的工程错误。

### 导师独立执行的数值验证（3/3 通过）

以下代码独立测试原始 1% 与改进 0.1% 假阳性率两种场景，并验证数值与自然频数方法一致。由导师 Python 环境运行通过。

In [ ]:
import math

prevalence = 0.0001
sensitivity = 0.99

def posterior_reference(prior, likelihood, false_positive_rate):
    numerator = prior * likelihood
    evidence = numerator + false_positive_rate * (1 - prior)
    return numerator / evidence

old_posterior = posterior_reference(prevalence, sensitivity, 0.01)
new_posterior = posterior_reference(prevalence, sensitivity, 0.001)

checks = {
    "original natural frequencies": math.isclose(old_posterior, 99/10098, rel_tol=1e-12),
    "improved natural frequencies": math.isclose(new_posterior, 990/10989, rel_tol=1e-12),
    "improved posterior increases": new_posterior > old_posterior
}
assert all(checks.values()), checks

print("Old false positive 1%:", old_posterior, "=", f"{old_posterior:.2%}")
print("New false positive 0.1%:", new_posterior, "=", f"{new_posterior:.2%}")
print(f"PASS {sum(checks.values())}/{len(checks)}")

## 第 3 讲：官方 Build It 第一步——亲手实现贝叶斯定理函数

已经理解公式、分母来源与假阳性率的作用，下一步由**学生独立写 Python**，而不是只看原作者实现：

```python
def bayes(prior, likelihood, false_positive_rate):
    # prior: P(D)，患病率
    # likelihood: P(+ | D)，检测灵敏度
    # false_positive_rate: P(+ | not D)，健康人假阳性率
    # 返回 P(D | +)
    pass
```

使用刚推导的完整二元公式

$$P(D\mid+)=\frac{P(+\mid D)\,P(D)}{P(+\mid D)\,P(D)+P(+\mid\neg D)(1-P(D))}.$$

学生提交后导师独立验证：
- 原版：`bayes(0.0001,0.99,0.01)` 约为 `0.0098039215686`；
- 改进版：`bayes(0.0001,0.99,0.001)` 约为 `0.09009009009`；
- 明确的特殊情况：`prior=1` 应得到 1（当检测灵敏度非零）；`prior=0` 应得到 0（当假阳性率非零）；
- 若证据概率分母为零，说明该阳性事件在模型假设下不可能出现，条件概率没有定义。可抛出 `ValueError`；不要把“后验=0”当作数学事实。

目前处于 Build It 阶段；尚未进入 01/07 正式 post quiz，不更新 `LEARNING.md` 的完成状态。

## 第 3 讲（验收）：学生的 Bayes 实现、边界辨析

### 学生独立提交的原始代码

学生正确写出了一个分子（真阳性的联合概率）除以完整证据概率（真阳性 + 假阳性）的计算流程。这一版本对应官方教程 Build It 第 1 步，且**不调用任何外部机器学习库**。

学生对于零分母的解释：“意味着没有这个检查到一个有症状的, 不能返回0”。

**评价**：“不能返回 0”正确；更严谨的说法是：分母 $P(+)=0$，表示按照当前概率模型，**检测阳性事件本身概率为零**，并不是“没有症状”（$+$ 代表检测阳性，不代表症状）。在基本条件概率定义 $P(D\mid+)=P(D\cap+)/P(+)$ 中，若 $P(+)=0$，条件概率**未定义**。

学生原代码在零分母时会自动抛 `ZeroDivisionError`，符合这一数学事实。为了给业务 API 提供清晰异常消息，可在明确需要时增加 `if evidence == 0: raise ValueError(...)`，但这属于导师推荐的增强，不替换学生正确的原始实现。

另外，先验为 0 并不一定导致零分母：若 $P(D)=0$ 且假阳性率 $P(+\mid\neg D)>0$，总体仍会出现假阳性，后验合法地等于 0。先验为 1、灵敏度非零时，后验合法地等于 1。

**导师已实际独立执行 13/13 项测试**，涵盖 1% 与 0.1% 假阳性率、先验边界、完美检测器、仅健康者出现阳性、后验范围、随着先验和假阳性率变化的单调性，以及三组零证据概率的 `ZeroDivisionError`。

### 学生原始实现（完整保留，不替换为导师修改版）

In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    # prior: P(D)
    # likelihood: P(+ | D)
    # false_positive_rate: P(+ | not D)
    # 返回 P(D | +)
    return (likelihood * prior) / (likelihood * prior + false_positive_rate * (1 - prior))

### 导师实际运行的测试：13/13（可复现）

In [ ]:
import math

bayes_test_results = {
    "official case": math.isclose(bayes(.0001,.99,.01),.000099/.010098,rel_tol=1e-12),
    "lower false positive rate": math.isclose(bayes(.0001,.99,.001),.000099/.0010989,rel_tol=1e-12),
    "zero prior with positive false positive": bayes(0,.99,.01)==0,
    "certain prior with positive likelihood": bayes(1,.99,.01)==1,
    "equal likelihood and FPR keeps prior": math.isclose(bayes(.32,.7,.7),.32,rel_tol=1e-12),
    "perfect detector": bayes(.14,1.,0.)==1.,
    "only healthy gives positive": bayes(.14,0.,.25)==0.,
    "posterior within [0,1]": all(0<=bayes(p,s,f)<=1 for p,s,f in [
        (.2,.4,.6),(.9,.6,.15),(.02,.99,.01)]),
    "lower FPR gives higher posterior": bayes(.0001,.99,.001)>bayes(.0001,.99,.01),
    "higher prior gives higher posterior": bayes(.5,.6,.1)>bayes(.1,.6,.1),
}

for i, args in enumerate([(0.,.99,0.),(1.,0.,.01),(.5,0.,0.)],start=1):
    try:
        bayes(*args)
    except ZeroDivisionError:
        bayes_test_results[f"zero evidence raises {i}"] = True
    else:
        bayes_test_results[f"zero evidence raises {i}"] = False

assert len(bayes_test_results)==13
assert all(bayes_test_results.values()), bayes_test_results
print(f"Bayes student's original implementation: {sum(bayes_test_results.values())}/13 passed")
print("Original FPR:",bayes(.0001,.99,.01))
print("Reduced FPR:",bayes(.0001,.99,.001))

## 第 4 讲：朴素贝叶斯（Naive Bayes），从“一个证据”扩展到“一封邮件里的许多词”

### 真实问题

有一批已标注为 spam（垃圾邮件）或 ham（正常邮件）的训练消息。对于一条新消息，例如 “free meeting”，我们想知道它应该属于哪个类别。

贝叶斯公式适用于“类别 $C$”与“邮件词特征 $w_1,\ldots,w_n$”：

$$P(C\mid w_1,\ldots,w_n)
=\frac{P(w_1,\ldots,w_n\mid C)P(C)}{P(w_1,\ldots,w_n)}.$$

难点在 $P(w_1,\ldots,w_n\mid C)$：直接统计每一种词序列组合，训练样本远远不够。

### 朴素性 = 类别已知时的条件独立假设

朴素贝叶斯假定 **给定类别 $C$ 后**，词特征按所选模型可视为条件独立，因此：

$$P(w_1,\ldots,w_n\mid C)\approx\prod_i P(w_i\mid C).$$

这不等于说“在所有邮件中，单词天然相互独立”，也不等于说“邮件类别和单词独立”。例如 “New York” 这种短语通常不独立，故称“朴素”假设。

对候选类别而言，分母 $P(w_1,\ldots,w_n)$ 相同，做分类 **argmax** 时可忽略：

$$\hat C=\arg\max_C\left[P(C)\prod_iP(w_i\mid C)\right].$$

数值实现改为对数域求和：

$$\boxed{\hat C=\arg\max_C\left[\log P(C)+\sum_i\log P(w_i\mid C)\right]}.$$

这直接复用了第 06 课学生自己学过的“对数把连乘变成加法”这一技巧。

### 训练时究竟要统计哪些数？

官方教程使用 **Multinomial Naive Bayes** 风格的 token 词频模型：

- `class_counts[C]`：某类别中有多少**封训练邮件**，用于估计 $P(C)$；
- `word_counts[C][w]`：该类别的训练邮件里单词 $w$ **总共出现多少次**，一个邮件重复出现两次就计两次；
- `class_word_totals[C]`：该类别所有训练邮件中的 token 总数；
- `vocab`：整个训练集出现过哪些不同单词。

估计类别先验：

$$P(C)=\frac{\mathrm{class\_counts}[C]}{\mathrm{total\_documents}}.$$

为什么还需要“拉普拉斯平滑”？若某词在某类训练数据里出现 0 次，直接用词频给它分配 0 概率，则对数会变成 $-\infty$。采用加 $\alpha$ 的平滑规则（$\alpha>0$）：

$$\boxed{P(w\mid C)=\frac{\mathrm{word\_counts}[C][w]+\alpha}
{\mathrm{class\_word\_totals}[C]+\alpha|\mathrm{vocab}|}}.$$

注意分母是**该类别全部 token 数**加上平滑质量，而不是该类别邮件的封数；这与简单统计“某词出现在多少封邮件里”属于不同特征模型。

### 先练训练阶段，不急着实现全部分类器

**任务**：你自己写 `NaiveBayes` 的初始化与 `train` 两个方法，实现文档数、词频、总词数、词表的统计。仅需 Python 标准库。

```python
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # self.smoothing
        # self.class_counts: 每个类的邮件数
        # self.word_counts: 每个类每个词的总出现次数
        # self.class_word_totals: 每个类的总 token 数
        # self.vocab: 训练集全部不同的词
        pass

    def train(self, documents, labels):
        # 对每一封邮件及其标签进行遍历
        # doc.lower().split() 得到词列表
        # 每处理一个 token，更新相应词频、类别总 token 数和词表
        pass
```

**小样本验证**：

```python
model = NaiveBayes()
model.train(
    ["free free win", "meeting schedule"],
    ["spam", "ham"]
)
assert model.class_counts["spam"] == 1
assert model.class_counts["ham"] == 1
assert model.word_counts["spam"]["free"] == 2
assert model.class_word_totals["spam"] == 3
assert model.class_word_totals["ham"] == 2
assert model.vocab == {"free", "win", "meeting", "schedule"}
```

**思考题**：在这个例子中，单词 “free” 只出现在第一封邮件，但累计出现 2 次。为什么朴素贝叶斯的词频模型会记录 `2` 而不是 `1`？“邮件中出现过这个词”与“这个词出现了多少次”分别对应什么样的特征？

收到你写的两个方法后，再实施真实测试并进入 `predict`、拉普拉斯平滑和概率对照。

## 第 4 讲：NaiveBayes 训练统计——学生代码验收

**学生原始回答**：“因为统计的是每一个单词出现的所有次数,如果不统计某个单词出现的总次数,这样我们就丢失了他们的总体的出现的比例,只会知道这个单词出现过”。

解释正确：多项式朴素贝叶斯记录 token 出现次数；伯努利特征只记录某个词是否在文档中出现。对于 `free free win`，多项式词频 `free=2`，而文档是否出现该词为 1。在这两封训练邮件组成的词表 $|V|=4$ 下，以 $\alpha=1$ 做加一平滑：

$$P(\mathrm{free}|\mathrm{spam})=(2+1)/(3+4)=3/7,\quad
P(\mathrm{free}|\mathrm{ham})=(0+1)/(2+4)=1/6.$$

**导师真实执行结果**：基础 7 项检查 **7/7 通过**：类别文档数 2 项、重复词次数、spam/ham token 总数、词表、smoothing 属性。

**与官方代码的预处理行为比较**：**0/3 通过**。官方 `code/bayes.py` 使用 `doc.lower().split()`；学生写的是 `words.split(' ')`，有以下区别：

1. `"Free free WIN"`：学生将三个词按大小写区分，官方统一小写后 `free=2`。
2. `"free  win"`：学生多统计了空字符串 `""`，错误地把 token 总数算成 3 而不是 2。
3. `"meeting\tschedule"`：学生未分开制表符，误将整段计为一个 token。

**本题状态：基础统计正确，文本规范化待修订。** 暂不提前给出改好的整个类，也不替学生修改代码。学生应独立在现有 `train` 中统一 `lower()` 并改用不带参数的 `split()`。

附加工程提醒：`zip(documents, labels)` 遇长度不等会静默截断，生产级输入校验可检查长度一致；这是额外建议，不是本练习的失败项。

### 学生原始代码（未经导师替换）

In [ ]:
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.split(' '):
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

### 可复现测试和学生修订要求

以下代码检查学生的原始版本，基础检查通过；官方预处理一致性检查**预计失败**，按字典逐项打印，避免未修订前整个 Notebook 在 `assert` 上中断。学生修订后可复用相同测试验证 3 项是否全部通过。

**只需修订分词逻辑**，不必重写已经正确的四类统计结构。修好后把完整的 `train` 方法发给导师进行重新验收，再实现 `predict`。

In [ ]:
base=NaiveBayes()
base.train(["free free win","meeting schedule"],["spam","ham"])
basic={
    "spam documents": base.class_counts["spam"]==1,
    "ham documents": base.class_counts["ham"]==1,
    "repeated free token": base.word_counts["spam"]["free"]==2,
    "spam total tokens": base.class_word_totals["spam"]==3,
    "ham total tokens": base.class_word_totals["ham"]==2,
    "vocabulary": base.vocab=={"free","win","meeting","schedule"},
    "smoothing stored": base.smoothing==1.0,
}
assert all(basic.values())
case=NaiveBayes()
case.train(["Free free WIN"],["spam"])
spaces=NaiveBayes()
spaces.train(["free  win"],["spam"])
tabs=NaiveBayes()
tabs.train(["meeting\tschedule"],["ham"])
edge={
    "lowercase": case.vocab=={"free","win"} and case.word_counts["spam"]["free"]==2,
    "consecutive spaces": spaces.vocab=={"free","win"} and spaces.class_word_totals["spam"]==2,
    "tab separator": tabs.vocab=={"meeting","schedule"} and tabs.class_word_totals["ham"]==2,
}
print("Basic:",sum(basic.values()),"/",len(basic))
print("Official tokenization:",sum(edge.values()),"/",len(edge))
print("Failing cases:",[name for name,passed in edge.items() if not passed])

## 第 4 讲修订验收：学生 NaiveBayes 初始化与训练统计通过（13/13）

学生独立修改了 `train()` 的分词表达式，从 `words.split(' ')` 改为 `words.lower().split()`。**符合官方 `code/bayes.py` 的预处理行为。**

导师在实际 Python 环境调用学生提交的完整代码，执行 **13 项测试，13/13 通过**：

- 原先的 7 项基础统计：spam、ham 文档数量；单词重复次数；spam、ham token 总数；训练词汇表；smoothing 属性。
- 原先不通过的 3 项官方分词一致性检查：统一大小写、连续空格、制表符分词。**现在 3/3 通过。**
- 另外补充的 3 项检查：含换行与混合空白的 token 分词、首尾空白不会生成空词、非默认 smoothing 参数正常保存。**3/3 通过。**

本次修订只修复预处理，保留学生自己选择的数据结构、原始注释和统计方式。

### 学生提出的新问题：“使用多项式模型及拉普拉斯平滑时这个是干什么的？”

**多项式朴素贝叶斯 Multinomial Naive Bayes** 关心某类别的训练材料里，单词共出现多少个 token，因此 `free free win` 中 `free` 的计数是 2 而不是 1。它使用类别条件 token 词频近似词分布，而不仅仅是判断“是否出现过这个词”。伯努利朴素贝叶斯的词存在特征则将该词视为 0/1。

**拉普拉斯平滑 Laplace Smoothing** 关心有限训练数据中从未见过的词。没有平滑时，某类训练中词计数为 0，估计的条件词概率就为 0，整个词概率的乘积可能被一个未见词归零；使用 log probability 时则出现 $\log 0$ 的问题。

对多项式 token 词频模型的有限训练词汇表 $V$：

$$P(w\mid C)=\frac{\mathrm{count}(w,C)+\alpha}{N_C+\alpha|V|},\qquad\alpha>0.$$

这里 $N_C$ 是类别 $C$ 中全部 token 的计数，$|V|$ 是**全训练集不同词的数量**。分母需要额外增加 $\alpha|V|$，因为词汇表里的每个词都获得了 $\alpha$ 伪计数，整个分布必须归一化；`train()` 阶段只存计数，不应在训练中反复修改计数来“添加平滑”。

### 具体计算

训练数据：

```python
documents = ["free free win", "meeting schedule"]
labels = ["spam", "ham"]
```

有 $N_{\mathrm{spam}}=3,\ N_{\mathrm{ham}}=2,\ |V|=4$。对于 $\alpha=1$：

| 单词 | spam 原始次数 | $P(w\mid\mathrm{spam})$ | ham 原始次数 | $P(w\mid\mathrm{ham})$ |
|---|---:|---:|---:|---:|
| free | 2 | $3/7$ | 0 | $1/6$ |
| win | 1 | $2/7$ | 0 | $1/6$ |
| meeting | 0 | $1/7$ | 1 | $2/6$ |
| schedule | 0 | $1/7$ | 1 | $2/6$ |

每个类别的四种词概率之和都是 1。$P(\mathrm{free}\mid\mathrm{spam})=3/7$ 与 $P(\mathrm{free}\mid\mathrm{ham})=1/6$ 的区别提供分类证据；这是后续 `predict()` 的基础。

对 `free meeting`，无平滑时两个类别的原始词似然乘积均为零；平滑后每个词在训练词汇表的每个类别里获得非零概率，因此可以有效比较。

**$\alpha$ 越大**，伪计数的影响越强，各词概率趋向更均匀；$\alpha$ 越接近 0，估计越接近直接使用原始词频。对于测试时不在训练词汇表内的 OOV 词，需要另行约定“忽略或映射至 UNK”等规则；直接套用训练词汇表的分母去给任意多个新词分配概率，不会构成归一化的词分布。这是后续工程扩展，不影响此处已知词的学习。

### 学生修订后的完整原始代码（严格保留）

In [ ]:
from collections import defaultdict

class NaiveBayes:
    def __init__(self, smoothing=1.0):
        # 初始化：
        # self.smoothing
        # self.class_counts
        # self.word_counts
        # self.class_word_totals
        # self.vocab
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        # 根据每封邮件及其标签更新上述统计量
        for words,label in zip(documents,labels):
            self.class_counts[label] += 1
            for word in words.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

### 导师实际运行的 13 项测试（修订后全通过）

In [ ]:
train_base=NaiveBayes()
train_base.train(["free free win","meeting schedule"],["spam","ham"])
train_checks={
    "spam doc count":train_base.class_counts["spam"]==1,
    "ham doc count":train_base.class_counts["ham"]==1,
    "repeated free token":train_base.word_counts["spam"]["free"]==2,
    "spam token total":train_base.class_word_totals["spam"]==3,
    "ham token total":train_base.class_word_totals["ham"]==2,
    "vocab":train_base.vocab=={"free","win","meeting","schedule"},
    "default smoothing":train_base.smoothing==1.,
}
train_case=NaiveBayes()
train_case.train(["Free free WIN"],["spam"])
train_spaces=NaiveBayes()
train_spaces.train(["free  win"],["spam"])
train_tabs=NaiveBayes()
train_tabs.train(["meeting\tschedule"],["ham"])
train_checks.update({
    "lowercase":train_case.vocab=={"free","win"} and train_case.word_counts["spam"]["free"]==2,
    "repeated spaces":train_spaces.vocab=={"free","win"} and train_spaces.class_word_totals["spam"]==2,
    "tabs":train_tabs.vocab=={"meeting","schedule"} and train_tabs.class_word_totals["ham"]==2,
})
train_extra=NaiveBayes(.5)
train_extra.train(["  FREE \n free\tWIN \n","meeting\t schedule"],["spam","ham"])
train_checks.update({
    "newlines":train_extra.word_counts["spam"]["free"]==2 and train_extra.word_counts["spam"]["win"]==1,
    "trim blanks":"" not in train_extra.vocab and train_extra.class_word_totals["spam"]==3,
    "custom smoothing attr":train_extra.smoothing==.5,
})
assert len(train_checks)==13 and all(train_checks.values()),train_checks
print(f"Student NaiveBayes training: {sum(train_checks.values())}/13 passed.")

## 下一步：检验自己对拉普拉斯平滑的理解，然后实现词的 log likelihood

训练阶段已通过。此时 `self.smoothing` **只是被保存的参数**，还没有作用于任何概率计算。

**先回答两个问题：**
1. 给定 `free free win`（spam）和 `meeting schedule`（ham），为什么用拉普拉斯平滑时分母需要加 $\alpha|V|$，而不只是加 $\alpha$？
2. 已知 $\alpha=1$、$|V|=4$，你能根据原始词频亲手算出 $P(\mathrm{meeting}\mid\mathrm{spam})$ 和 $P(\mathrm{meeting}\mid\mathrm{ham})$ 吗？

若已掌握，请进一步**独立实现** `_log_likelihood(self,word,cls)`，返回 $\ln P(word\mid cls)$，并从 `self.word_counts`、`self.class_word_totals`、`len(self.vocab)` 和 `self.smoothing` 读取数据。后续将使用同一方法编写 `predict()`。

## 拉普拉斯平滑：学生理解核验

**学生原话**：“我们要保证整体的概率和为1”。

**判定：正确。** 对每个固定类别 C，需要保证训练词汇表 V 上单词条件概率之和为 1：

$$\sum_{w\in V} P(w\mid C)=1.$$

每个词增加一个平滑伪计数 alpha，整个词表的计数总共增加 alpha × |V|：

$$\sum_{w\in V}[\operatorname{count}(w,C)+\alpha]=N_C+\alpha|V|.$$

举例：训练数据 free free win 属于 spam；词表包括 free、win、meeting、schedule。原始计数分别 2、1、0、0，alpha=1 后变成 3、2、1、1，总和由 3 增为 7；分母若是 7，概率和为 1；分母若错误地写成 4，概率和为 7/4，违反归一化。

**学习进度：** 学生已理解多项式词频统计与拉普拉斯平滑的必要性；下一步计算具体条件词概率，再编写 log likelihood；尚未完成课后测验。

## 练习：计算不同类别的平滑条件概率（等待作答）

训练两封邮件：spam 类别的内容是 free free win；ham 类别的内容是 meeting schedule。

完整训练词汇表由 free、win、meeting、schedule 四个词组成，即 |V|=4。采用 alpha=1。

请分别计算：

1. $P(\mathrm{meeting}\mid\mathrm{spam})$；
2. $P(\mathrm{meeting}\mid\mathrm{ham})$。

写出每一项的分子和分母，并解释为何两个类别的分母可能不同。

此题通过后由学生独立实现 _log_likelihood(self, word, cls)，使用学习过的平滑公式和自然对数。